In [ ]:
import pandas as pd
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator #type: ignore
from tensorflow.keras.models import Sequential #type: ignore
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.optimizers import Adam

# --- 1. Load your DataFrame ---
# Assume it has columns: 'filepath' and 'label'
df = pd.read_csv('image_labels.csv')  # or construct manually

# Example structure:
df = pd.DataFrame({
'filepath': ['images/img1.jpg', 'images/img2.jpg'],
'label': ['cat', 'dog']
})

# --- 2. Split the DataFrame ---
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

# --- 3. Setup ImageDataGenerator ---
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label',
    target_size=(128, 128),
    class_mode='binary',
    batch_size=32,
    shuffle=True
)

val_gen = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filepath',
    y_col='label',
    target_size=(128, 128),
    class_mode='binary',
    batch_size=32,
    shuffle=False
)

# --- 4. Build a simple CNN ---
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# --- 5. Train the model ---
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)